# Multi-Agent Orchestration 


## 전체 흐름

| 패턴 | 누가 다음 작업자를 결정하는가 | 특징 |
|---|---|---|
| 직접 만든 Sub-agent | Supervisor 에이전트 | 가장 단순하고 커스터마이즈 쉬움 |
| Supervisor | 중앙 supervisor | 역할 분배와 최종 종합 |
| Handoff | supervisor + worker | worker끼리 직접 위임 |
| Swarm | 현재 active agent | peer-to-peer 협업 |

예제 팀 구성:

| 역할 | 책임 |
|---|---|
| `market_researcher` | 시장 조사, 타깃 고객, 경쟁 구도 |
| `product_strategist` | 핵심 기능, 포지셔닝, MVP 범위 |
| `financial_analyst` | 수익 모델, 가격, 비용 구조 |
| `risk_reviewer` | 리스크, 실행 난이도, 검토 |

공통 프롬프트:

> AI 기반 영어 회화 앱의 MVP 기획안을 만들어줘. 타깃 고객, 핵심 기능, 수익 모델, 리스크를 포함해.


## 환경 준비


```text
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=mtvs2026-multi-agent
```


In [19]:
from dotenv import load_dotenv

load_dotenv()


True

## 1. 직접 만든 Sub-agent


### 1.1. Worker 에이전트 2명 만들기


In [20]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

llm = init_chat_model('openai:gpt-5.4-mini')

market_researcher = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 스타트업 시장 조사 담당자. 타깃 고객, 경쟁 구도, 시장 기회를 짧고 구체적으로 정리해.'
)

product_strategist = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 SaaS/앱 제품 전문가. MVP 핵심 기능과 포지셔닝을 명확하게 제안해'
)

### 1.2. Worker를 도구로 감싸기


In [21]:
@tool
def ask_market_researcher(prompt: str) -> str:
    """시장 규모, 타깃 고객, 경쟁 구도 조사가 필요할 때 호출, prompt에 자세한 요구사항을 넣어줘."""
    result = market_researcher.invoke({"messages": [HumanMessage(prompt)]})
    return result["messages"][-1].content


@tool
def ask_product_strategist(prompt: str) -> str:
    """제품 포지셔닝, MVP 기능, 차별화 전략이 필요할 때 호출"""
    result = product_strategist.invoke({"messages": [HumanMessage(prompt)]})
    return result["messages"][-1].content


### 1.3. Supervisor 에이전트


In [22]:
supervisor = create_agent(
    model=llm,
    tools=[ask_market_researcher, ask_product_strategist],
    system_prompt='''너는 스타트업 사업기획 리드.
    
    원칙:
    1. 사용자 요청을 받으면 어떤 전문가가 필요한지 먼저 판단
    2. 시장, 고객,경쟁은 ask_market_researcher에 위임
    3. MVP 기능, 포지셔닝은 ask_product_strategist에 위임
    4. 두 전문가의 답을 합쳐서 실행 가능한 MVP 기획안을 요약    
    '''
)


### 1.4. 복합 요청 실행


In [23]:
result = supervisor.invoke({
    "messages": [HumanMessage(
        "AI 기반 영어 회화 앱의 MVP 기획안을 만들어주세요."
        "타깃 고객, 핵심 기능, 포지셔닝을 포함해서."
    )]
})

print(result["messages"][-1].content)

아래는 시장/고객/경쟁, 제품 전략을 합쳐 정리한 **AI 기반 영어 회화 앱 MVP 기획안**입니다.

---

# 1) MVP 한 줄 정의
**“사용자가 30초 안에 시작해, 5분 안에 영어로 실제 대화를 해보고, 대화 후 바로 교정과 복습까지 할 수 있는 AI 영어 회화 앱”**

---

# 2) 타깃 고객
## 1차 타깃: 취업준비생/대학생
**추천 이유**
- 영어면접, 오픽, 토스 등 **목표가 명확**
- 짧은 기간에 반복 연습이 필요해 **재방문 동기**가 강함
- AI의 강점인 **즉시 피드백, 반복 대화, 표현 교정**과 잘 맞음
- 가격 민감하지만 앱 결제에는 익숙함

## 2차 타깃
### 직장인
- 해외업무, 미팅, 이메일, 발표 준비 등 실전 수요가 큼
- 다만 업무 상황별 개인화 요구가 높아 MVP 초기에는 조금 무거움

### 해외여행/유학 준비자
- 단기 니즈가 명확하고 진입 장벽이 낮음
- 다만 지속 사용보다는 단기 사용 성격이 강함

---

# 3) 핵심 고객 pain point
- 영어를 알아도 **실제로 말이 안 나옴**
- 화상영어는 **시간/비용 부담**이 큼
- 혼자 연습하면 **피드백이 부족**
- 초보자는 영어로만 시작하면 **진입장벽이 높음**
- “내 상황에 맞는 말”을 배우고 싶지만 기존 앱은 **실전 맥락이 약함**

---

# 4) 경쟁 대안과 한계
## 화상영어/전화영어
- 장점: 사람과 실시간 대화 가능
- 한계: 예약 부담, 비용 부담, 개인화/즉시 피드백 약함

## 영어 학습 앱
- 장점: 저렴하고 접근성 높음
- 한계: 말하기 훈련이 약하고 “알지만 못 말하는 문제”를 못 해결

## 1:1 과외/튜터
- 장점: 맞춤형
- 한계: 비싸고 확장성이 낮음

## AI 대화앱
- 장점: 24시간 가능, 부담 적음
- 한계: 대화가 반복되면 질림, 성과 체감이 약하면 이탈

---

# 5) 포지셔닝
## 추천 포지셔닝
**“영어를 배우는 앱”이 아니라 “매일 5분, 바로 말하게 만드는 AI 영어

### 1.5. 왜 이 패턴이 좋은가

| 한 에이전트가 다 함 | Supervisor + Worker |
|---|---|
| 시장, 제품, 수익, 리스크를 한 프롬프트에 모두 넣음 | 역할별 worker 프롬프트를 작게 유지 |
| 조사, 기능 설계, 검토가 한 메시지 흐름에 모두 누적 | supervisor는 worker 결과만 받아 종합 |
| 토큰 비용과 역할 혼선 증가 | 토큰 절약과 역할 분리 |

직접 만든 Sub-agent 패턴은 구조가 단순해서 원리를 설명하기 좋습니다. 다만 worker 수가 늘어나면 도구 래퍼와 라우팅 규칙을 직접 관리해야 합니다.


## [실습]
1. `art_director` worker 등 추가 (이미지 프롬프트 생성 전문, 디자인 전문 등).
2. supervisor 가 어느 worker 를 부르는지 LangSmith trace 에서 확인.
3. Worker 에 도구를 각자 다르게 (lore_writer 에 wiki 검색, balance 에 spreadsheet 계산) 붙이기.


---

In [35]:
art_director = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 UI/UX 디자인 전문가. 사용자 경험을 고려한 디자인과 인터페이스를 제안해.'
)

@tool
def ask_art_director(prompt: str) -> str:
    """디자인, UI/UX 관련 질문이 필요할 때 호출"""
    result = art_director.invoke({"messages": [HumanMessage(prompt)]})
    return result["messages"][-1].content

supervisor_1 = create_agent(
    model=llm,
    tools=[ask_market_researcher, ask_product_strategist, ask_art_director],
    system_prompt='''너는 스타트업 사업기획 리드.
    
    원칙:
    1. 사용자 요청을 받으면 어떤 전문가가 필요한지 먼저 판단
    2. 시장, 고객, 경쟁은 ask_market_researcher에 위임
    3. MVP 기능, 포지셔닝은 ask_product_strategist에 위임
    4. 디자인, UI/UX는 ask_art_director 위임
    5. 세 전문가의 답을 합쳐서 실행 가능한 MVP 기획안을 요약    
    '''
)

result_1 = supervisor_1.invoke({
    "messages": [HumanMessage(
        "AI 기반 영상 편집 앱의 MVP 기획안을 만들어주세요."
        "타깃 고객, 핵심 기능, 포지셔닝을 포함해서."
    )]
})

print(result_1["messages"][-1].content)


아래는 세 전문가 관점을 합친 **AI 기반 영상 편집 앱 MVP 기획안**입니다.

---

# 1) 제품 한 줄 정의

**“긴 영상이나 촬영본을 몇 분 만에 숏폼/SNS용 콘텐츠로 바꿔주는 한국어 친화 AI 영상 편집 앱”**

핵심은 “편집을 잘하는 툴”이 아니라  
**“업로드 가능한 결과물을 가장 빨리 만들어주는 재가공 엔진”**입니다.

---

# 2) 타깃 고객

## 1순위 타깃: 숏폼 크리에이터
- 유튜브 쇼츠, 틱톡, 릴스 제작자
- 1인 크리에이터, 성장 초기 계정 운영자
- 편집 빈도가 높고 반복 사용 가능성이 큼

## 2순위 타깃: 소상공인/자영업자
- 매장 홍보, 상품 소개, 이벤트 영상이 필요한 사람
- 편집 인력 없이도 콘텐츠가 필요한 경우가 많음
- “쉽게, 빠르게, 그럴듯하게”에 반응이 큼

## 보조 타깃
- 1인 마케터, 소규모 브랜드 팀
- 교육/코치/지식 콘텐츠 제작자

## 우선순위 결론
1. **숏폼 크리에이터**
2. **소상공인/자영업자**
3. 마케터/브랜드 실무자

---

# 3) 고객 pain point

## 공통 pain point
- 편집 시간이 너무 오래 걸림
- 툴이 어렵고 배워야 할 게 많음
- 결과물이 “초보 티”가 남
- 플랫폼별 규격 맞추기가 번거로움
- AI 자동화가 있어도 결국 손볼 게 많음

## 숏폼 크리에이터의 pain point
- 긴 영상에서 쓸 구간 찾기 어려움
- 자막, 컷 분할, 리사이즈 반복 작업이 피곤함
- 매일/자주 올려야 해서 속도가 중요

## 소상공인의 pain point
- 무엇을 찍고 어떻게 편집할지 모름
- 편집자 없이 홍보 영상을 만들어야 함
- 촬영본은 있는데 홍보용 결과물로 바꾸기 어려움

---

# 4) 포지셔닝

## 추천 포지셔닝
**“긴 영상을 SNS용 숏폼으로 자동 변환하는 AI 편집 앱”**

또는 더 실용적으로:
**“편집 지식 없이도 바로 올릴 수 있는 영상 콘텐츠를 빠르게 만드는 앱”**

## 포지셔닝 원칙
- 범용 영상 편집기

## 2. Supervisor 패턴

- supervisor + worker 를 직접 손으로 만들어봤다면, **`langgraph-supervisor`** 라이브러리는 이 패턴을 한 줄로 묶어줍니다.

### 2.1. Worker 두 명 만들기


In [24]:
llm = init_chat_model("openai:gpt-5.4-mini")


@tool
def search_market(keyword: str) -> str:
    """스타트업 시장 조사 자료 검색. 데모용 가짜 데이터."""
    db = {
        "영어 회화": "성인 직장인과 취업 준비생의 회화 학습 수요가 높음. 기존 앱은 반복 학습과 실제 대화 지속률이 약점.",
        "AI 튜터": "개인화 피드백, 발음 교정, 상황극 대화가 핵심 차별화 포인트.",
        "경쟁 앱": "Duolingo, Speak, Cambly 등이 존재. 가격, 실시간성, 개인화 수준에서 차별화 필요.",
    }
    return db.get(keyword, "관련 시장 자료 없음")


@tool
def estimate_unit_economics(monthly_price: int, users: int, churn_rate: float) -> str:
    """월 구독 가격, 유료 사용자 수, 월 이탈률 기준 간단한 수익 모델 계산."""
    mrr = monthly_price * users
    retained_users = int(users * (1 - churn_rate))
    return f"MRR {mrr:,}원, 다음 달 예상 잔존 유저 {retained_users:,}명, 월 이탈률 {churn_rate:.1%}"

In [25]:
market_researcher = create_agent(
    model=llm,
    tools=[search_market],
    system_prompt='너는 스타트업 시장 조사 담당자. 타깃 고객, 경쟁 구도, 시장 기회를 짧고 구체적으로 정리해.',
    name='market_researcher'
)

product_strategist = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 제품 전문가. MVP 핵심 기능과 포지셔닝, 우선순위를 명확하게 제안해',
    name='produect_strategist'
)

financial_analyst = create_agent(
    model=llm,
    tools=[estimate_unit_economics],
    system_prompt='너는 재무 분석가. 수익 모델, 가격, 간단한 unit economics를 숫자로 정리해',
    name='financial_analyst'
)

### 2.2. Supervisor 한 줄로 만들기


In [27]:
%pip install langgraph_supervisor
from langgraph_supervisor import create_supervisor

supervisor_workflow = create_supervisor(
    agents=[market_researcher, product_strategist, financial_analyst],
    model=llm,
    prompt=(
        "너는 스타트업 사업기획 리드. 사용자 요청을 전문가에게 적절히 분배해.\n"
        "- 시장, 고객, 경쟁은 market_researcher \n"
        "- MVP 기능, 포지셔닝은 product_strategist \n"
        "- 수익 모델, 가격은 financial_analyst\n"
        "전문가 답을 종합해 실행 가능한 MVP 기획안을 작성."
    )
)

team = supervisor_workflow.compile()


Note: you may need to restart the kernel to use updated packages.


### 2.3. 복합 요청 실행


In [28]:
result = team.invoke({
    'messages': [HumanMessage(
        "AI 기반 영어 회화 앱의 MVP 기획안을 만들어주세요."
        "타깃 고객, 핵심 기능, 수익 모델을 포함해서."
    )]
})

print(result['messages'][-1].content)

아래는 **AI 기반 영어 회화 앱 MVP 기획안**입니다.  
핵심은 **“혼자서도 매일 말하게 만드는 초경량 회화 코치”**로 잡는 것입니다.

---

## 1) 타깃 고객

### 1순위: 20~35세 직장인/취준생
- **니즈**: 영어 회화 실력은 필요하지만, 학원 갈 시간은 부족
- **문제**: 말할 기회가 적고, 틀릴까 봐 부담됨
- **구매 가능성**: 높음, 자기계발 지출에 익숙함

### 2순위: 해외여행/이직 준비자
- **니즈**: 짧은 기간에 실전 회화가 필요
- **문제**: 문법보다 “바로 말하는 연습”이 필요
- **구매 가능성**: 이벤트성 수요가 강함

### 3순위: 영어 말하기 초중급 학습자
- **니즈**: 단어/문법은 아는데 회화가 안 됨
- **문제**: 원어민과 대화할 기회 부족
- **구매 가능성**: 반복 사용 가능성이 높음

---

## 2) 핵심 기능

### MVP 필수 기능
1. **AI 대화 연습**
   - 주제 선택: 자기소개, 여행, 면접, 직장 회의 등
   - 음성 또는 텍스트 입력
   - AI가 대화 상대 역할 수행

2. **즉시 피드백**
   - 문법 오류, 자연스러운 표현, 더 쉬운 표현 제안
   - 사용자의 답변을 짧고 명확하게 교정
   - “이 문장 이렇게 말하면 더 자연스러움” 형태로 제공

3. **대화 리캡/복습**
   - 자주 틀린 표현 저장
   - 오늘의 표현 3개 복습
   - 대화 종료 후 요약 리포트 제공

### 있으면 좋은 기능
- 발음 점수
- 레벨 테스트
- 목표별 코스(여행/면접/비즈니스)
- 하루 5분 학습 알림

---

## 3) 수익 모델

### 1. 구독 모델
- **Free**: 하루 대화 횟수 제한, 기본 피드백만 제공
- **Pro 월 구독**: 무제한 대화, 고급 피드백, 복습 기능
- 가장 안정적이고 MVP에 적합

### 2. 프리미엄 기능 과금
- 면접 영어, 비즈니스 영어, 여행 영어 같은 **전문 시나리오 팩**
- 발음

### 2.4. 누가 무엇을 답했는지 trace


In [29]:
print(f"총 메시지 수: {len(result['messages'])}\n")
for i, m in enumerate(result["messages"][-8:]):
    name = getattr(m, "name", None) or type(m).__name__
    content = (m.content if isinstance(m.content, str) else str(m.content))[:80]
    print(f"  [{i}] {name}: {content}")

총 메시지 수: 7

  [0] HumanMessage: AI 기반 영어 회화 앱의 MVP 기획안을 만들어주세요.타깃 고객, 핵심 기능, 수익 모델을 포함해서.
  [1] supervisor: 
  [2] transfer_to_market_researcher: Successfully transferred to market_researcher
  [3] market_researcher: 아래는 **AI 기반 영어 회화 앱 MVP 기획안**입니다.  
핵심은 **“혼자서도 매일 말하게 만드는 초경량 회화 코치”**로 잡는 것입니다
  [4] market_researcher: Transferring back to supervisor
  [5] transfer_back_to_supervisor: Successfully transferred back to supervisor
  [6] supervisor: 아래는 **AI 기반 영어 회화 앱 MVP 기획안**입니다.  
핵심은 **“혼자서도 매일 말하게 만드는 초경량 회화 코치”**로 잡는 것입니다


### 2.5. Supervisor가 자동으로 하는 일

| 단계 | 동작 |
|---|---|
| 1 | 사용자 메시지를 받음 |
| 2 | 어떤 worker가 적합한지 판단(도구 호출 형식으로)  |
| 3 | 그 worker에게 메시지 전달 |
| 4 | worker 결과를 보고 추가 worker 호출 여부 결정 |
| 5 | 충분하면 최종 답변 합성 |

- `create_supervisor(agents=[...], model=llm, prompt=...)`는 직접 만든 Sub-agent 패턴의 반복 코드를 줄여줍니다. 
- worker마다 `name`이 필요하고, `.compile()` 후 일반 LangGraph처럼 `invoke()` 또는 `stream()`으로 실행합니다.


## [실습]
1. supervisor 가 worker 를 몇 번 호출했는지 messages 의 `name` 필드로 카운트.
2. LangSmith trace 에서 worker 별 토큰 비용 분리해서 보기.

---

## 3. Handoff 패턴

- Supervisor 패턴은 모든 결정이 중앙(supervisor)에 모입니다. 반대로 **Handoff** 는 한 에이전트가 "이건 너가 맡아" 하면서 직접 다른 에이전트에 넘기는 방식.
- `create_handoff_tool` 로 "넘기기 도구" 를 만들어 worker 끼리 주고받게 합니다.

### 3.1. Handoff 도구 만들기


In [36]:
from langgraph_supervisor import create_handoff_tool

# 각 worker에서 다른 worker로 넘기는 handoff 도구를 부여
handoff_to_market = create_handoff_tool(
    agent_name="market_researcher",
    description="시장 조사, 타깃 고객, 경쟁 분석이 필요할 때 호출."
)

handoff_to_product = create_handoff_tool(
    agent_name="product_strategist",
    description="MVP 기능, 포지셔닝, 제품 우선순위가 필요할 때 호출."
)

handoff_to_finance = create_handoff_tool(
    agent_name="finance_analyst",
    description="수익 모델, 가격, 비용 구조 계싼이 필요할 때 호출."
)

handoff_to_risk = create_handoff_tool(
    agent_name="risk_reviewer",
    description="기획안 검토, 리스크, 실행 난이도 평가가 필요할 때 호출."
)

### 3.2. 3명의 에이전트, 서로 위임 가능
시장 조사 담당자 ↔ 제품 전략가 ↔ 재무 분석가 ↔ 리스크 검토자가 서로에게 일을 넘길 수 있게 도구를 부여.

In [37]:
llm = init_chat_model("openai:gpt-5.4-mini")


@tool
def estimate_subscription_revenue(monthly_price: int, paid_users: int) -> str:
    """월 구독 가격과 유료 사용자 수로 MRR을 계산."""
    return f"예상 MRR: {monthly_price * paid_users:,}원"


market_researcher = create_agent(
    model=llm,
    tools=[handoff_to_product, handoff_to_finance, handoff_to_risk],
    system_prompt=(
        "너는 시장 조사 담당자. 타깃 고객과 경쟁 구도를 정리한 뒤 다음 단계가 필요한지 판단.\n"
        "- 제품 기능이 필요하면 product_strategist로 handoff\n"
        "- 수익 모델이 필요하면 financial_analyst로 handoff\n"
        "- 검토가 필요하면 risk_reviewer로 handoff"
    ),
    name="market_researcher",
)

product_strategist = create_agent(
    model=llm,
    tools=[handoff_to_market, handoff_to_finance, handoff_to_risk],
    system_prompt=(
        "너는 제품 전략가. MVP 기능과 포지셔닝을 정리한 뒤 다음 단계를 결정.\n"
        "- 시장 근거가 부족하면 market_researcher\n"
        "- 가격·수익 모델이 필요하면 financial_analyst\n"
        "- 리스크 검토가 필요하면 risk_reviewer"
    ),
    name="product_strategist",
)

financial_analyst = create_agent(
    model=llm,
    tools=[estimate_subscription_revenue, handoff_to_market, handoff_to_product, handoff_to_risk],
    system_prompt=(
        "너는 재무 분석가. 가격, 수익 모델, 비용 구조를 숫자로 정리.\n"
        "- 시장 가정이 부족하면 market_researcher\n"
        "- 기능 범위가 불명확하면 product_strategist\n"
        "- 검토가 필요하면 risk_reviewer"
    ),
    name="financial_analyst",
)

risk_reviewer = create_agent(
    model=llm,
    tools=[handoff_to_market, handoff_to_product, handoff_to_finance],
    system_prompt=(
        "너는 리스크 검토자. 기획안의 빈 곳과 실행 리스크를 짚고 부족하면 담당자에게 다시 handoff.\n"
        "충분하면 '검토 완료'라고 짧게 마무리."
    ),
    name="risk_reviewer",
)


### 3.3. Supervisor가 첫 진입을 정하고, 이후 worker handoff 보기

- Supervisor 가 `lore_writer` 로 첫 위임을 한 번 하고, 그 뒤로는 worker 내부의 handoff tool 이 다음 담당을 직접 지정합니다.


In [38]:
from langgraph_supervisor import create_supervisor

workflow = create_supervisor(
    agents=[market_researcher, product_strategist, financial_analyst, risk_reviewer],
    model=llm,
    prompt="첫 진입자는 market_researcher. 이후 worker들이 필요한 전문가에게 직접 넘기게 둬."
)

team = workflow.compile()

### 3.4. 실행, 누가 누구에게 넘기는지 보기

- LLM 이 서로에게만 넘기는 무한 핑퐁을 막기 위해 `recursion_limit` 을 함께 지정합니다.


In [39]:
result = team.invoke({
    "messages":[HumanMessage(
        "AI 기반 영어 회화 앱의 MVP 기획안을 만들어줘."
        "타깃 고객, 핵심 기능, 수익 모델, 리스크를 포함해."
    )]
})

print(result['messages'][-1].content)

아래는 **AI 기반 영어 회화 앱의 MVP 기획안**입니다.  
실행 가능한 수준으로 **타깃 고객, 핵심 기능, 수익 모델, 리스크**를 중심으로 정리했습니다.

---

# AI 기반 영어 회화 앱 MVP 기획안

## 1) 서비스 개요
**AI와 영어로 대화하며, 사용자의 말하기 습관과 실전 회화 능력을 짧고 반복적으로 개선하는 앱**

핵심은 “영어 공부”보다  
**“매일 말하게 만드는 것”**입니다.

---

## 2) 타깃 고객

### 1차 타깃
**영어 회화에 자신감이 부족한 20~40대 한국인 성인**
- 문법/독해는 어느 정도 가능하지만 말하기가 약한 사용자
- 학원, 과외, 전화영어는 비용·시간 부담이 큰 사용자
- 혼자 연습할 상대가 없어 지속이 어려운 사용자

### 세부 타깃
1. **직장인**
   - 업무 이메일/문서는 가능하지만 회화는 약함
   - 회의, 출장, 간단한 스몰토크 수요

2. **취준생**
   - 영어 면접, 자기소개, 상황별 답변 연습 필요

3. **여행 준비자**
   - 공항, 호텔, 식당, 쇼핑 등 실전 표현만 빠르게 배우고 싶음

### 왜 이 타깃인가
- 영어 회화의 가장 큰 장벽은 **실전 대화 기회 부족**
- AI는 1:1 무제한 연습에 강점이 있어 니즈와 잘 맞음
- 짧은 시간에 반복 사용이 가능한 형태로 설계하기 좋음

---

## 3) 해결하려는 문제
사용자가 영어 회화를 못 하는 이유는 주로 다음과 같습니다.

1. **말할 기회가 부족하다**
2. **틀릴까 봐 두렵다**
3. **무엇을 말해야 할지 모르겠다**
4. **피드백을 받아도 복습이 이어지지 않는다**

MVP는 이 네 가지 문제를 해결하는 데 집중해야 합니다.

---

## 4) 핵심 기능

## A. AI 대화 세션
### 기능
- 사용자가 주제를 선택하면 AI가 영어로 대화를 시작
- 텍스트 또는 음성 기반 대화 제공
- 난이도 선택 가능: 초급 / 중급

### 예시 주제
- 자기소개
- 카페 주문
- 여행


### 3.5. Supervisor vs Handoff, 언제 무엇을 쓰는가

| 상황 | 권장 |
|---|---|
| 작업 흐름이 예측 가능하고 중앙 통제가 필요함(분기 적음) | Supervisor |
| worker끼리 다음 담당자를 동적으로 정해야 함(누가 다음인지 case-by-case) | Handoff |
| 단순 분배와 최종 종합이 중요함 | Supervisor |
| 전문가들이 서로 핑퐁하며 협의해야 함 | Handoff |

- Handoff는 유연하지만 무한 위임이 생길 수 있습니다. `recursion_limit`을 항상 함께 보여주는 편이 안전합니다.


## [실습]

1. handoff 도구의 description 을 더 엄격하게 적고 호출 빈도 변화 관찰.
2. recursion_limit 을 5 로 낮춰 무한 핑퐁 방지 (LLM 이 서로에게만 넘기는 경우).
3. handoff 시점에 LangSmith metadata 에 "reason" 을 남기도록.


---

## 4. Swarm 패턴
- `langgraph-swarm` 은 supervisor 없이 "누가 활성 에이전트인가" 를 동적으로 바꿔가며 진행. peer-to-peer 협업이 필요할 때.

### 4.1. Worker들, Swarm용 handoff 도구

- Swarm에서는 현재 활성 에이전트가 다른 에이전트로 직접 handoff합니다. 이 예제에서는 일반 도구 호출과 handoff 도구 호출이 한 응답에서 섞여 OpenAI 메시지 형식 오류가 나지 않도록 Swarm 전용 LLM에서 병렬 도구 호출을 끕니다.


In [42]:
%pip install langgraph_swarm
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_openai import ChatOpenAI
from langgraph_swarm import create_swarm, create_handoff_tool as create_swarm_handoff_tool

swarm_llm: BaseChatModel = ChatOpenAI(
    model='gpt-5.4-mini',
    model_kwargs={'parallel_tool_calls': False}
)

swarm_to_market = create_swarm_handoff_tool(
    agent_name="swarm_market_researcher",
    description="시장 조사 담당자에게 넘김."
)

swarm_to_product = create_swarm_handoff_tool(
    agent_name="swarm_product_strategist",
    description="제품 전략가에게 넘김."
)

swarm_to_finance = create_swarm_handoff_tool(
    agent_name="swarm_financial_analyst",
    description="재무 분석가에게 넘김."
)

swarm_to_risk = create_swarm_handoff_tool(
    agent_name="swarm_risk_reviewer",
    description="리스크 검토자에게 넘김."
)

Note: you may need to restart the kernel to use updated packages.


In [43]:
@tool
def estimate_mrr(monthly_price: int, paid_users: int) -> str:
    """월 구독 가격과 유료 사용자 수로 MRR을 계산."""
    return f"MRR = {monthly_price * paid_users:,}원"

In [44]:
swarm_market_researcher = create_agent(
    model=swarm_llm,
    tools=[swarm_to_product, swarm_to_finance, swarm_to_risk],
    system_prompt=(
        "너는 시장 조사 담당자. 타깃 고객과 경쟁 구도를 정리해. "
        "제품 전략이 필요하면 swarm_product_strategist에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_market_researcher",
)

swarm_product_strategist = create_agent(
    model=swarm_llm,
    tools=[swarm_to_market, swarm_to_finance, swarm_to_risk],
    system_prompt=(
        "너는 제품 전략가. 핵심 기능과 포지셔닝을 정리해. "
        "수익 모델이 필요하면 swarm_financial_analyst에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_product_strategist",
)

swarm_financial_analyst = create_agent(
    model=swarm_llm,
    tools=[estimate_mrr, swarm_to_market, swarm_to_product, swarm_to_risk],
    system_prompt=(
        "너는 재무 분석가. 가격과 수익 모델을 숫자로 정리해. "
        "검토가 필요하면 swarm_risk_reviewer에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_financial_analyst",
)

swarm_risk_reviewer = create_agent(
    model=swarm_llm,
    tools=[swarm_to_market, swarm_to_product, swarm_to_finance],
    system_prompt=(
        "너는 리스크 검토자. 타깃 고객, 기능, 수익 모델, 리스크가 모두 있으면 '검토 완료'로 마무리해. "
        "부족한 부분이 있으면 해당 담당자에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_risk_reviewer",
)

### 4.2. Swarm 만들기, 기본 활성 에이전트 지정


In [45]:
swarm_workflow = create_swarm(
    agents=[
        swarm_market_researcher,
        swarm_product_strategist,
        swarm_financial_analyst,
        swarm_risk_reviewer        
    ],
    default_active_agent="swarm_market_researcher"
)

# 멀티턴에서 마지막 active_agent를 이어가려면 checkpointer를 추가함
# from langgraph.checkpoint.memory import InMemorySaver
# swarm = swarm_workflow.compile(checkpointer=InMemorySaver())

swarm = swarm_workflow.compile()

### 4.3. 실행


In [46]:
result = swarm.invoke(
    {
        "messages":[
            HumanMessage(
                "기업 임직원용 AI 영어 회화 코칭 서비스를 만들려고 한다. "
                "B2B 타깃 고객, 핵심 기능, 가격 정책, 도입 리스크를 정리해줘."
            )
        ]
    },
    config={"recursion_limit":18}
)

In [47]:
print("=== Active agent 시퀀스 ===")
for message in result["messages"]:
    name = getattr(message, "name", None)
    if name:
        print(f"  -> {name}")

print("\n마지막 active_agent:", result.get("active_agent"))
print("\n최종:", result["messages"][-1].content[:300])

=== Active agent 시퀀스 ===
  -> swarm_market_researcher
  -> transfer_to_swarm_product_strategist
  -> swarm_product_strategist

마지막 active_agent: swarm_product_strategist

최종: 아래는 **기업 임직원용 AI 영어 회화 코칭 서비스** 기준으로 정리한 **B2B 타깃 고객, 핵심 기능, 가격 정책, 도입 리스크**입니다.  
(참고: 수익 모델/정교한 가격 산정은 별도 재무 분석이 필요하면 더 깊게 볼 수 있습니다.)

---

## 1) B2B 타깃 고객

### 1순위 타깃
**글로벌 커뮤니케이션 수요가 높은 중견·대기업**
- 해외 법인/지사와 협업이 잦은 기업
- 해외 고객, 파트너, 공급망과 영어 커뮤니케이션이 많은 조직
- 임원/팀장/실무자가 영어 미팅, 이메일, 발표를 자주 하는 회사

**대표


### 4.4. Supervisor / Handoff / Swarm 한눈에

| 패턴 | 누가 결정 | 라이브러리 |
|---|---|---|
| Supervisor | supervisor 한 명 | `langgraph-supervisor` |
| Supervisor + handoff 도구 | supervisor + worker | `langgraph-supervisor` |
| Swarm | 활성 에이전트 본인 | `langgraph-swarm` |

- Swarm은 supervisor가 없고 **어느 에이전트가 활성인지**를 state로 관리합니다.
- `default_active_agent`가 시작점이고, `active_agent`가 다음 턴의 시작점을 결정합니다. 현재 예제는 일반 도구와 handoff 도구가 동시에 호출되어 메시지 짝이 깨지는 문제를 피하기 위해 `parallel_tool_calls=False`를 사용합니다.


### 정리

- Sub-agent는 자식 에이전트를 도구처럼 호출해 역할과 컨텍스트를 분리합니다.
- 직접 만든 Sub-agent 패턴은 원리를 설명하기 좋지만, worker가 늘어나면 래퍼와 라우팅 관리가 복잡해집니다.
- `langgraph-supervisor`는 중앙 사업기획 리드가 worker 호출과 최종 종합을 맡는 구조에 적합합니다.
- Handoff는 worker끼리 다음 담당자를 직접 정해야 하는 동적 협업에 유용합니다.
- Swarm은 supervisor 없이 active agent가 이동하는 peer-to-peer 구조입니다.
- 동적 협업 구조는 무한 핑퐁이 생길 수 있으므로 `recursion_limit`과 trace 확인이 중요합니다.


### [실습]

1. handoff 도구 description 을 더 좁히면 호출 패턴이 어떻게 변하는지.
2. 활성 에이전트가 무한 핑퐁할 때 `recursion_limit` 으로 멈추기.
3. swarm 의 state 에 마지막 active_agent 가 어떻게 저장되는지 확인 (`result.get("active_agent")`).
